In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType, StructType, StructField

# workspace.silver.dim_song
dim_song = spark.table("workspace.bronze.excel_song_catalog").select("song_id", "original_artist", "song_title")
dim_song.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.dim_song")
valid_song_ids_count = dim_song.count()
print(f"dim_song: {valid_song_ids_count} rows")

dim_song: 699 rows


In [0]:
spotify_raw = spark.table("workspace.bronze.api_spotify")

track_schema = ArrayType(StructType([
    StructField("id", StringType()),
    StructField("name", StringType()),
    StructField("artists", ArrayType(StructType([StructField("name", StringType())]))),
    StructField("album", StructType([
        StructField("name", StringType()),
        StructField("release_date", StringType()),
    ])),
    StructField("external_ids", StructType([StructField("isrc", StringType())])),
]))
 
spotify_parsed = (
    spotify_raw
    .withColumn("items", F.from_json("raw_json", track_schema))
    .withColumn("item", F.explode_outer("items"))
    .select(
        "song_id",
        F.upper(F.col("item.external_ids.isrc")).alias("isrc"),
        F.upper(F.col("item.name")).alias("recording_title"),
        F.upper(F.concat_ws(", ", F.col("item.artists.name"))).alias("artist_name"),
        F.upper(F.col("item.album.name")).alias("album"),
        F.col("item.album.release_date").alias("release_date"),
        F.col("item.id").alias("spotify_track_id"),
        F.when(F.col("queried_with_artist"), "high").otherwise("low").alias("match_confidence"),
    )
    .filter(F.col("isrc").isNotNull())
    .dropDuplicates(["song_id", "isrc"])
)

spotify_parsed = (
    spotify_parsed
    .join(dim_song.select("song_id", "song_title"), "song_id", "left")
    .withColumn("recording_title_base", F.trim(F.regexp_replace("recording_title", r"\s*\(.*\)\s*$", "")))
    .withColumn("song_title_clean", F.trim(F.upper("song_title")))
    .filter(F.col("recording_title_base") == F.col("song_title_clean"))
    .drop("recording_title_base", "song_title_clean", "song_title")
)
 
spotify_parsed.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.fact_song_isrc")
print(f"fact_song_isrc: {spotify_parsed.count()} rows")

fact_song_isrc: 705 rows


In [0]:
isrc_counts = spotify_parsed.groupBy("song_id").agg(F.count("*").alias("isrc_count"))
song_isrc_count = (
    dim_song.select("song_id")
    .join(isrc_counts, "song_id", "left")
    .fillna({"isrc_count": 0})
)
song_isrc_count.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.song_isrc_count")
print(f"song_isrc_count: {song_isrc_count.count()} rows")

song_isrc_count: 699 rows


In [0]:
if spark.catalog.tableExists("workspace.bronze.api_youtube"):
    youtube_raw = spark.table("workspace.bronze.api_youtube")

    video_schema = ArrayType(StructType([
        StructField("id", StructType([StructField("videoId", StringType())])),
        StructField("snippet", StructType([
            StructField("channelId", StringType()),
            StructField("channelTitle", StringType()),
            StructField("title", StringType()),
            StructField("publishedAt", StringType()),
        ])),
    ]))

    youtube_parsed = (
        youtube_raw
        .withColumn("items", F.from_json("raw_json", video_schema))
        .withColumn("item", F.explode_outer("items"))
        .select(
            "song_id",
            F.col("item.id.videoId").alias("video_id"),
            F.col("item.snippet.channelId").alias("channel_id"),
            F.col("item.snippet.channelTitle").alias("channel_title"),
            F.col("item.snippet.title").alias("video_title"),
            F.col("item.snippet.publishedAt").alias("published_at"),
        )
        .filter(F.col("video_id").isNotNull())
        .dropDuplicates(["song_id", "video_id"])
    )

    youtube_parsed.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.fact_song_youtube_video")
    print(f"fact_song_youtube_video: {youtube_parsed.count()} rows")

    video_counts = youtube_parsed.groupBy("song_id").agg(F.count("*").alias("video_count"))
    song_video_count = (
        dim_song.select("song_id")
        .join(video_counts, "song_id", "left")
        .fillna({"video_count": 0})
    )
    song_video_count.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.song_video_count")
    print(f"song_video_count: {song_video_count.count()} rows")
else:
    print("workspace.bronze.api_youtube not found yet -- skipping YouTube silver tables. "
          "Run ingest_api_youtube first, then re-run this notebook.")

fact_song_youtube_video: 1209 rows
song_video_count: 699 rows
